# **Correcciones post-EDA: outliers monetarios, mercado real y diseno de modelado**

Continuacion del EDA sobre el dataset limpio. Motivado por 3 hallazgos de la corrida anterior:

1. **Outlier absurdo**: una adjudicacion de ~2.1x10^17 COP (~140 veces el PIB anual de
   Colombia) via la entidad "EAG" destruye el HHI general (9,995), el Pareto
   ("1 proveedor = 80% del valor") y la serie de valor mensual.
2. **`adjudicado` estructuralmente vacio en modalidades no competitivas**: regimen
   especial y contratacion directa (82% del dataset) tienen 0.00% exacto — el campo
   no se llena para esas modalidades. El modelo de probabilidad de adjudicacion debe
   restringirse al universo competitivo.
3. **Fuga de informacion**: `respuestas_al_procedimiento` (corr 0.78 con adjudicado)
   se conoce solo DESPUES del cierre; no es feature valida al momento de publicacion.

Este notebook: (A) hace forense del outlier y define una regla de plausibilidad
monetaria, (B) rehace el analisis de mercado con valores creibles, (C) investiga los
picos de enero 2022 y enero 2026, (D) deja definidos los universos y features del
modelado para la capacidad 3.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
# Cargar archivos


df_lineas = pd.read_csv('secop_ctei_lineas_limpio.csv', low_memory=False)
df_proc = pd.read_csv('secop_ctei_procesos_limpio.csv', low_memory=False)

In [3]:


warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DTYPE = {"nit_entidad": "string", "nit_del_proveedor_adjudicado": "string"}
df_lineas = pd.read_csv("secop_ctei_lineas_limpio.csv", dtype=DTYPE, low_memory=False)
df_proc = pd.read_csv("secop_ctei_procesos_limpio.csv", dtype={"nit_entidad": "string"}, low_memory=False)

for col in ["fecha_de_publicacion_del", "fecha_adjudicacion"]:
    for d in (df_lineas, df_proc):
        if col in d.columns:
            d[col] = pd.to_datetime(d[col], errors="coerce")

df_lineas["adjudicado_bool"] = df_lineas["adjudicado"].map({"Si": True, "No": False}).astype("boolean")
print(f"lineas: {len(df_lineas):,} | procesos: {len(df_proc):,}")

lineas: 495,284 | procesos: 492,797


# **A. Forense del outlier monetario y regla de plausibilidad**

## A.1 Ver las lineas mas grandes con contexto completo

Referencia de escala: el Presupuesto General de la Nacion es del orden de 5x10^14 COP
(~500 billones) al anio. Cualquier linea individual por encima de eso es fisicamente
imposible como contrato real.

In [4]:
top_lineas = df_lineas.nlargest(15, "valor_total_adjudicacion")[[
    "id_del_proceso", "entidad", "nombre_del_proveedor",
    "nombre_del_procedimiento", "modalidad_de_contratacion",
    "precio_base", "valor_total_adjudicacion", "fecha_de_publicacion_del", "urlproceso"
]]
top_lineas

,id_del_proceso,entidad,nombre_del_proveedor,nombre_del_procedimiento,modalidad_de_contratacion,precio_base,valor_total_adjudicacion,fecha_de_publicacion_del,urlproceso
281869,CO1.REQ.6849316,EAG,CORPORACION CONSTRULET S.A.S,REALIZAR CONSULTORIA PARA LA ELABORACIÓN DE ES...,Contratación régimen especial (con ofertas),463454668,214790230013979506,2024-09-16,https://community.secop.gov.co/Public/Tenderin...
431529,CO1.REQ.9483601,MINISTERIO DE MINAS Y ENERGIA,GECELCA S.A. E.S.P.,Administrar los recursos y gestionar proyectos...,Contratación Directa (con ofertas),4205027751839,4205027751839,2025-12-28,https://community.secop.gov.co/Public/Tenderin...
392211,CO1.REQ.8781496,PATRIMONIO AUTÓNOMO AEROCAFÉ,CONSORCIO AEROPUERTO DEL CAFE SK,CONVOCATORIA ABIERTA CONTRATACION OBRA LADO AI...,Contratación régimen especial (con ofertas),639583680291,634275135745,2025-08-25,https://community.secop.gov.co/Public/Tenderin...
475428,CO1.REQ.10324110,Empresa Distrital de Desarrollo y Renovación U...,CONSORCIO PLANTA CURVAL,INVITACION PUBLICA A OFERTAR DE MAYOR CUANTIA,Contratación régimen especial (con ofertas),861875326914,410758596204,2026-04-06,https://community.secop.gov.co/Public/Tenderin...
209227,CO1.REQ.5622810,MINISTERIO DE AGRICULTURA Y DESARROLLO RURAL,FONDO PARA EL FINANCIAMIENTO DEL SECTOR AGROPE...,CONTRATO INTERADMINISTRATIVO FINAGRO,Contratación Directa (con ofertas),392463605046,392463605046,2024-01-25,https://community.secop.gov.co/Public/Tenderin...
393873,CO1.REQ.8794819,DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNO...,EMPRESA DE DESARROLLO URBANO DE MEDELLIN,Contrato interadministrativo de mandato sin re...,Contratación Directa (con ofertas),379790887521,379790887521,2025-08-28,https://community.secop.gov.co/Public/Tenderin...
200636,CO1.REQ.5441709,AEROCIVIL,ENTerritorio S.A,Realizar la gerencia integral del proyecto par...,Contratación Directa (con ofertas),363763308569,363763308569,2023-12-20,https://community.secop.gov.co/Public/Tenderin...
260544,CO1.REQ.6474584,MUNICIPIO DE PEREIRA- OFICIAL,TEK SOLUCIONES TECNOLOGICAS S.A.S,SUMINISTRO DE LICENCIAS DE OFFICE PARA COLEGIO...,Selección abreviada subasta inversa,336502412,335128000000,2024-07-05,https://community.secop.gov.co/Public/Tenderin...
87374,CO1.REQ.3758486,FONDO FINANCIERO DISTRITAL DE SALUD..,"AGENCIA DISTRITAL PARA LA EDUCACIÓN SUPERIOR, ...",Fortalecimiento de las capacidades en salud de...,Contratación Directa (con ofertas),334710179630,334710179630,2022-12-23,https://community.secop.gov.co/Public/Tenderin...
427460,CO1.REQ.9409106,MINISTERIO DE MINAS Y ENERGIA,EMPRESA DISTRIBUIDORA DEL PACIFICO S.A. E.S.P,CE- MAICAO; ALBANIA Y MUNICIPIOS ALEDAÑOS,Contratación Directa (con ofertas),322709746890,322709746890,2025-12-10,https://community.secop.gov.co/Public/Tenderin...


In [5]:
# Cuantas lineas superan umbrales de plausibilidad crecientes
umbral_tabla = pd.DataFrame({
    "umbral_cop": [1e11, 1e12, 1e13, 1e14, 5e14],
    "descripcion": ["100 mil millones", "1 billon", "10 billones", "100 billones",
                     "~Presupuesto Gral. Nacion anual"],
})
umbral_tabla["lineas_que_superan"] = [
    (df_lineas["valor_total_adjudicacion"] > u).sum() for u in umbral_tabla["umbral_cop"]
]
umbral_tabla["valor_acumulado_de_esas_lineas"] = [
    df_lineas.loc[df_lineas["valor_total_adjudicacion"] > u, "valor_total_adjudicacion"].sum()
    for u in umbral_tabla["umbral_cop"]
]
umbral_tabla

,umbral_cop,descripcion,lineas_que_superan,valor_acumulado_de_esas_lineas
0,"100,000,000,000.00",100 mil millones,41,214802740890141180
1,"1,000,000,000,000.00",1 billon,2,214794435041731345
2,"10,000,000,000,000.00",10 billones,1,214790230013979506
3,"100,000,000,000,000.00",100 billones,1,214790230013979506
4,"500,000,000,000,000.00",~Presupuesto Gral. Nacion anual,1,214790230013979506


## A.2 Regla de plausibilidad (dos criterios combinados)

- **Absoluto**: valor de linea > 1x10^13 COP (10 billones) es implausible como
  adjudicacion individual en este universo (servicios de ingenieria/consultoria/
  educacion — no megaobras tipo metro, que ademas irian por otros segmentos UNSPSC).
- **Relativo**: valor > 100x el precio_base cuando precio_base > 1'000,000 COP
  (un precio base serio). Captura errores de digitos aun por debajo del umbral absoluto.

Las lineas marcadas NO se borran: se excluyen del analisis monetario con
`flag_valor_implausible`, documentando cuantas son y cuanto "valor" fantasma aportan.

In [6]:
UMBRAL_ABSOLUTO = 1e13
RATIO_MAX = 100

flag_abs = df_lineas["valor_total_adjudicacion"] > UMBRAL_ABSOLUTO
flag_rel = (
    (df_lineas["precio_base"] > 1e6)
    & (df_lineas["valor_total_adjudicacion"] > RATIO_MAX * df_lineas["precio_base"])
)
df_lineas["flag_valor_implausible"] = flag_abs | flag_rel

n_flag = df_lineas["flag_valor_implausible"].sum()
valor_fantasma = df_lineas.loc[df_lineas["flag_valor_implausible"], "valor_total_adjudicacion"].sum()
valor_total_bruto = df_lineas["valor_total_adjudicacion"].sum()

print(f"Lineas marcadas implausibles: {n_flag:,} "
      f"({n_flag / len(df_lineas):.3%} de las lineas)")
print(f"'Valor' que aportaban: {valor_fantasma:,.0f} COP "
      f"= {valor_fantasma / valor_total_bruto:.1%} del valor total bruto")
print()
print("Muestra de lineas marcadas:")
df_lineas[df_lineas["flag_valor_implausible"]].nlargest(10, "valor_total_adjudicacion")[[
    "entidad", "nombre_del_proveedor", "precio_base", "valor_total_adjudicacion"
]]

Lineas marcadas implausibles: 14 (0.003% de las lineas)
'Valor' que aportaban: 214,791,136,543,172,512 COP = 100.0% del valor total bruto

Muestra de lineas marcadas:


,entidad,nombre_del_proveedor,precio_base,valor_total_adjudicacion
281869,EAG,CORPORACION CONSTRULET S.A.S,463454668,214790230013979506
260544,MUNICIPIO DE PEREIRA- OFICIAL,TEK SOLUCIONES TECNOLOGICAS S.A.S,336502412,335128000000
362160,INSTITUTO FINANCIERO PARA EL DESARROLLO DEL VA...,C Y C SOLUCIONES INTEGRALES SAS,150000000,150000000000
489488,CVC,ALCALDIA MUNICIPAL DE GINEBRA,99070693,99070693000
273926,COMANDO GAULA MILITARES (COGAM),CENTROS RECREACIONALES Y SEDES HABITACIONALES ...,80000000,79957000000
288102,ALCALDIA DISTRITAL BARRANCABERMEJA,SEMPRO,68835000,68835000000
471819,ALCALDIA MUNICIPIO DE NOCAIMA,INCEGER,43200000,43200000000
180270,ESE HOSPITAL OCTAVIO OLIVARES,"SERVICIOS, SUMINISTROS Y MONTAJES S.A.S",43200000,43020000000
424510,ALCALDÍA MUNICIPAL DE SOPO,NETSOLUTIONS S.A.S,43048250,34000000000
38699,MUNICIPIO SANTA ROSA DE OSOS,RUIZ & ZAPATA S.A.S,18600000,18600000000


## A.3 Hallazgos — outliers

- **Solo 14 lineas (0.003% del total) explican el 100.0% del "valor" bruto adjudicado**
  segun la marca `flag_valor_implausible`. No es un factor de escala uniforme (no es
  "todo esta multiplicado por 10^7"): es un puñado de filas puntuales con montos
  fisicamente imposibles. El caso EAG (214.79 x10^15 COP) es, con enorme diferencia, el
  peor: su `precio_base` (463 millones) es coherente con una consultoria normal, pero
  `valor_total_adjudicacion` esta ~463,000 veces por encima del precio base — clarisimo
  error de captura/parseo del dato fuente, no un contrato real reescalado.
- El resto de las 14 lineas marcadas (Municipio de Pereira 335 mil millones, IFV
  150 mil millones, CVC 99 mil millones, COGAM 80 mil millones, etc.) son de una escala
  muchisimo mas chica que EAG pero igual implausibles frente a su `precio_base`
  (razon > 100x) — sugiere que el problema de captura de `valor_total_adjudicacion` no es
  exclusivo de una sola entidad/proveedor, sino un patron recurrente de baja frecuencia en
  la fuente (SECOP) que conviene seguir vigilando en futuras cargas del dataset.
- La regla combinada (absoluta 10^13 COP + relativa 100x precio_base) es razonable: es
  conservadora en el umbral absoluto (deja pasar contratos grandes pero fisicamente
  posibles, ej. Ministerio de Minas y Energia con 4.2 billones de precio_base) y agresiva
  en el umbral relativo para atrapar errores de digitos que no llegan al umbral absoluto.
  Las lineas se **excluyen del analisis monetario, no se corrigen/reescalan** — decision
  correcta dado que no hay forma confiable de inferir el valor real original.
- Pendiente para reportar: documentar estas 14 lineas (con `id_del_proceso` y `urlproceso`)
  como candidatas a validar manualmente contra el portal SECOP si hay tiempo, ya que
  siguen contando como "procesos adjudicados" validos en el conteo de Capacidad 1/3 —
  solo se excluyeron de los calculos *monetarios*, correctamente.

# **B. Analisis de mercado corregido (sin valores implausibles)**

In [7]:
df_mercado = df_lineas[
    (df_lineas["adjudicado_bool"] == True)
    & (df_lineas["valor_total_adjudicacion"] > 0)
    & (~df_lineas["flag_valor_implausible"])
].copy()
df_mercado["proveedor_id"] = df_mercado["nit_del_proveedor_adjudicado"].fillna(df_mercado["nombre_del_proveedor"])

def hhi(part): return (part ** 2).sum() * 10000

mercado = (
    df_mercado.groupby("proveedor_id")["valor_total_adjudicacion"].sum()
    .sort_values(ascending=False).reset_index()
)
mercado["participacion"] = mercado["valor_total_adjudicacion"] / mercado["valor_total_adjudicacion"].sum()
mercado["participacion_acum"] = mercado["participacion"].cumsum()

hhi_corregido = hhi(mercado["participacion"])
n_80 = (mercado["participacion_acum"] <= 0.80).sum() + 1
print(f"HHI corregido: {hhi_corregido:,.1f} "
      f"({'baja' if hhi_corregido < 1500 else 'moderada' if hhi_corregido < 2500 else 'alta'} concentracion)")
print(f"Proveedores que concentran el 80% del valor: {n_80:,} de {len(mercado):,} ({n_80/len(mercado):.1%})")

top20 = mercado.head(20).sort_values("valor_total_adjudicacion")
fig = go.Figure(go.Bar(x=top20["valor_total_adjudicacion"], y=top20["proveedor_id"].astype(str), orientation="h"))
fig.update_layout(title="Top 20 proveedores por valor adjudicado (corregido)",
                   xaxis_title="Valor adjudicado (COP)", template="plotly_white", height=600)
fig.show()

HHI corregido: 116.1 (baja concentracion)
Proveedores que concentran el 80% del valor: 1,044 de 17,179 (6.1%)


In [8]:
df_mercado["fecha_de_publicacion_del"] = pd.to_datetime(
    df_mercado["fecha_de_publicacion_del"],
    errors="coerce"
)

# Serie mensual de valor adjudicado corregida (la del EDA anterior estaba dominada por el outlier)
serie_valor = (
    df_mercado[df_mercado["fecha_de_publicacion_del"].notna()]
    .assign(anio_mes=lambda d: d["fecha_de_publicacion_del"].dt.to_period("M").dt.to_timestamp())
    .groupby("anio_mes")["valor_total_adjudicacion"].sum().reset_index()
)
fig = go.Figure(go.Scatter(x=serie_valor["anio_mes"], y=serie_valor["valor_total_adjudicacion"], mode="lines"))
fig.update_layout(title="Valor adjudicado por mes (sin implausibles)",
                   yaxis_title="COP", template="plotly_white")
fig.show()

In [9]:
# HHI por entidad, corregido, con las mismas condiciones (min 20 procesos adjudicados)
def hhi_por_entidad(df):
    filas = []
    for ent, sub in df.groupby("entidad"):
        val = sub.groupby("proveedor_id")["valor_total_adjudicacion"].sum()
        filas.append({"entidad": ent, "hhi": hhi(val / val.sum()),
                       "proveedores": sub["proveedor_id"].nunique(),
                       "procesos": sub["id_del_proceso"].nunique(),
                       "valor_total": val.sum()})
    return pd.DataFrame(filas)

hhi_ent = hhi_por_entidad(df_mercado)
print("Top 15 entidades por valor (corregido):")
display(hhi_ent.sort_values("valor_total", ascending=False).head(15))

filtradas = hhi_ent[hhi_ent["procesos"] >= 20].sort_values("hhi", ascending=False)
print("\nEntidades mas concentradas (min 20 procesos adjudicados):")
display(filtradas.head(10))
print("\nEntidades mas competitivas:")
display(filtradas.tail(10))

Top 15 entidades por valor (corregido):


,entidad,hhi,proveedores,procesos,valor_total
779,DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNO...,"1,383.75",264,949,5727377498700
1611,MINISTERIO DE MINAS Y ENERGIA,"7,610.81",68,87,4836052632118
1560,INVIAS,163.98,656,972,1631509329408
301,ANI,329.50,80,78,1316704294502
698,DEPARTAMENTO DE ANTIOQUIA//,"1,909.81",177,279,1238997511450
1607,MINISTERIO DE EDUCACION NACIONAL (MEN),"1,521.10",84,148,1092231185004
14,"AGENCIA DISTRITAL PARA LA EDUCACIÓN SUPERIOR, ...",714.58,69,54,1001244950027
1603,MINISTERIO DE AGRICULTURA Y DESARROLLO RURAL,"6,904.40",29,35,997418818827
2089,SECRETARIA DE EDUCACION DEL DISTRITO,"1,067.58",118,151,963028590183
6,AEROCIVIL,"1,996.42",203,241,836833475956



Entidades mas concentradas (min 20 procesos adjudicados):


,entidad,hhi,proveedores,procesos,valor_total
1083,FABRICA DE LICORES Y ALCOHOLES DE ANTIOQUIA,"9,150.34",37,50,103290566604
2125,SENA REGIONAL ANTIOQUIA Grupo de Apoyo Adminis...,"9,005.77",28,38,25791067148
1075,Empresa Distrital de Desarrollo y Renovación U...,"8,706.73",27,29,440396888176
59,ALCALDIA DE GIRARDOTA,"8,668.72",10,20,31932303879
1755,MUNICIPIO DE MARINILLA,"8,569.09",14,53,42172124813
1731,MUNICIPIO DE ITAGUI,"8,184.94",9,54,516973344747
1696,MUNICIPIO DE EL DOVIO,"8,019.47",2,22,327509705
1611,MINISTERIO DE MINAS Y ENERGIA,"7,610.81",68,87,4836052632118
2238,UAE SETP AVANTE PASTO,"7,590.42",17,23,19320053364
1673,MUNICIPIO DE CHIQUINQUIRA+,"7,546.07",21,33,46339928902



Entidades mas competitivas:


,entidad,hhi,proveedores,procesos,valor_total
345,ASOCIACION DE MUNICIPIOS CORPORACIÓN AGENCIA P...,403.96,70,160,675592325
2102,SECRETARIA DISTRITAL DEL HABITAT-,402.64,44,35,49062967835
1451,INSTITUTO DE DESARROLLO URBANO,400.15,120,137,665621756467
186,ALCALDIA MUNICIPIO DE ARAUCA,336.58,85,123,20115146824
301,ANI,329.50,80,78,1316704294502
71,ALCALDIA DE PASTO,317.81,71,107,8798583784
1165,GOBIERNO DEPARTAMENTAL,308.91,176,253,139169075198
1144,GOBERNACION DEL HUILA*,220.98,117,145,50612487385
851,EMPRESA DE DESARROLLO URBANO DE MEDELLIN,195.50,108,282,123014079676
1560,INVIAS,163.98,656,972,1631509329408


## B.1 Hallazgos — mercado corregido

- **El diagnostico de mercado cambia por completo al quitar el outlier**: HHI general
  pasa de 9,995.4 ("monopolio") a **116.1 ("baja concentracion")** — esta ultima cifra es
  la que hay que citar en el reporte. El "1 proveedor = 80% del valor" del EDA original
  era enteramente un artefacto: corregido, se necesitan **1,044 de 17,179 proveedores
  (6.1%)** para llegar al 80% del valor, un mercado CTeI razonablemente competido a nivel
  agregado nacional.
- EAG desaparece del top 15 de entidades por valor una vez corregido — confirma que su
  presencia anterior era 100% producto del outlier, no de actividad real relevante.
- **El panorama por entidad individual (no agregado) si mostraba concentracion real,
  independiente del outlier**: Ministerio de Minas y Energia (HHI 7,610.81, valor real
  4.8 billones) y Patrimonio Autonomo Aerocafe (HHI 9,391.22, pero solo 2 procesos/2
  proveedores — mas señal de bajo volumen que de mercado cerrado) ya aparecian
  concentrados en el EDA original y se mantienen igual tras la correccion (no dependian
  del outlier EAG). Nuevas entidades mas concentradas identificadas aqui: Fabrica de
  Licores y Alcoholes de Antioquia (9,150), SENA Regional Antioquia (9,006), Empresa
  Distrital de Desarrollo (8,707) — todas con relativamente pocos proveedores (<40) y
  procesos (20-50), coherente con mercados locales/nicho mas que con anomalias de dato.
  <mark>Actualizacion (`Capacidad1_cierre_final.ipynb`, seccion 2):</mark> el caso
  Ministerio de Minas y Energia queda explicado — es el mayor de los 109 "fondos
  administrados" identificados alli (contrato con GECELCA S.A. E.S.P., dic-2025,
  ~4.4 billones reales, un unico proveedor por diseno). No es una rareza sin resolver:
  es un patron legitimo de administracion de fondos en el sector energetico/publico, y
  como tal debe reportarse como categoria aparte del "mercado competido", no como un caso
  de concentracion problematica a corregir.
- **INVIAS sigue siendo la entidad mas competitiva** (HHI 163.98, 656 proveedores
  distintos, 972 procesos) tanto antes como despues de la correccion — es un buen caso de
  referencia de "mercado sano" para contrastar en el reporte contra los casos concentrados.
- Conclusion para el reporte: **el mercado CTeI colombiano es competitivo a nivel
  agregado, pero con concentracion real y no trivial en entidades/sectores especificos**
  (energia, licores/alcoholes departamentales, SENA regional) — el mensaje correcto no es
  "hay monopolio" (falso, era el outlier) ni "todo es perfectamente competido" (tampoco
  es cierto, hay nichos concentrados de forma legitima, varios de ellos explicados por
  fondos administrados de un solo operador).

## B.2 Evolución de la participación de los proveedores

Se compara la participación de cada proveedor en el valor total adjudicado durante 2024 y 2025.

No se utiliza 2026 porque corresponde a un año incompleto. La variación se expresa en puntos porcentuales y permite identificar proveedores entrantes, salientes y aquellos que ganaron o perdieron participación.

In [10]:
# Comparación de participación de proveedores: 2024 vs. 2025

ANIO_INICIAL = 2024
ANIO_FINAL = 2025

base_participacion = df_mercado[
    df_mercado["fecha_de_publicacion_del"]
    .dt.year.isin([ANIO_INICIAL, ANIO_FINAL])
].copy()

base_participacion["anio"] = (
    base_participacion["fecha_de_publicacion_del"].dt.year
)

# Valor adjudicado por proveedor y año
participacion_anual = (
    base_participacion
    .groupby(["anio", "proveedor_id"], as_index=False)
    ["valor_total_adjudicacion"]
    .sum()
)

# Participación dentro del valor total de cada año
participacion_anual["participacion_pct"] = (
    participacion_anual["valor_total_adjudicacion"]
    / participacion_anual.groupby("anio")[
        "valor_total_adjudicacion"
    ].transform("sum")
    * 100
)

# Convertir los años en columnas
cambio_proveedores = (
    participacion_anual
    .pivot(
        index="proveedor_id",
        columns="anio",
        values="participacion_pct"
    )
    .fillna(0)
    .reset_index()
)

# Nombre visible del proveedor
nombres = (
    base_participacion
    .groupby("proveedor_id")["nombre_del_proveedor"]
    .first()
)

cambio_proveedores["proveedor"] = (
    cambio_proveedores["proveedor_id"]
    .map(nombres)
    .fillna(cambio_proveedores["proveedor_id"].astype(str))
)

# Cambio en puntos porcentuales
cambio_proveedores["cambio_pp"] = (
    cambio_proveedores[ANIO_FINAL]
    - cambio_proveedores[ANIO_INICIAL]
)

# Clasificación
cambio_proveedores["clasificacion"] = np.select(
    [
        (cambio_proveedores[ANIO_INICIAL] == 0)
        & (cambio_proveedores[ANIO_FINAL] > 0),

        (cambio_proveedores[ANIO_INICIAL] > 0)
        & (cambio_proveedores[ANIO_FINAL] == 0),

        cambio_proveedores["cambio_pp"] > 0
    ],
    [
        "Entrante",
        "Saliente",
        "Ganó participación"
    ],
    default="Perdió participación"
)

# Proveedores con mayores aumentos y disminuciones
ganadores = cambio_proveedores.nlargest(10, "cambio_pp")
perdedores = cambio_proveedores.nsmallest(10, "cambio_pp")

print("PROVEEDORES QUE MÁS GANARON PARTICIPACIÓN")
display(
    ganadores[
        ["proveedor", ANIO_INICIAL, ANIO_FINAL,
         "cambio_pp", "clasificacion"]
    ]
)

print("PROVEEDORES QUE MÁS PERDIERON PARTICIPACIÓN")
display(
    perdedores[
        ["proveedor", ANIO_INICIAL, ANIO_FINAL,
         "cambio_pp", "clasificacion"]
    ]
)

# Gráfica
datos_grafica = (
    pd.concat([ganadores, perdedores])
    .drop_duplicates("proveedor_id")
    .sort_values("cambio_pp")
)

fig = go.Figure(
    go.Bar(
        x=datos_grafica["cambio_pp"],
        y=datos_grafica["proveedor"],
        orientation="h",
        text=datos_grafica["cambio_pp"].map(
            lambda x: f"{x:+.2f} pp"
        ),
        textposition="outside",
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Cambio de participación: %{x:.2f} puntos porcentuales"
            "<extra></extra>"
        )
    )
)

fig.update_layout(
    title="Proveedores que ganaron o perdieron participación: 2024–2025",
    xaxis_title="Cambio en participación de mercado (puntos porcentuales)",
    yaxis_title="",
    height=750,
    template="plotly_white"
)

fig.show()

PROVEEDORES QUE MÁS GANARON PARTICIPACIÓN


anio,proveedor,2024,2025,cambio_pp,clasificacion
1142,GECELCA S.A. E.S.P.,0.00,23.15,23.15,Entrante
5803,CONSORCIO AEROPUERTO DEL CAFE SK,0.00,3.49,3.49,Entrante
438,EMPRESA DISTRIBUIDORA DEL PACIFICO S.A. E.S.P,0.00,1.78,1.78,Entrante
1649,AGENCIA NACIONAL INMOBILIARIA VIRGILIO BARCO V...,0.00,1.49,1.49,Entrante
9046,UT Gestión Integral Centros de Datos 2025,0.00,1.23,1.23,Entrante
965,ESU,0.83,1.78,0.96,Ganó participación
6349,CONSORCIO PROTECCION CARTAGENA,0.00,0.91,0.91,Entrante
4411,EMPRESA NACIONAL ESTATAL DE DESARROLLO TERRITO...,0.00,0.89,0.89,Entrante
4584,CONSORCIO ME CAROLINA,0.00,0.56,0.56,Entrante
5009,DESARROLLO CONJUNTO TERRITORIAL - DETERCONSA,0.00,0.51,0.51,Entrante


PROVEEDORES QUE MÁS PERDIERON PARTICIPACIÓN


anio,proveedor,2024,2025,cambio_pp,clasificacion
153,FONDO PARA EL FINANCIAMIENTO DEL SECTOR AGROPE...,4.42,0.99,-3.43,Perdió participación
132,FINANCIERA DE DESARROLLO TERRITORIAL S.A.,1.71,0.08,-1.63,Perdió participación
223,EMPRESA DE DESARROLLO URBANO DE MEDELLIN,5.86,4.28,-1.58,Perdió participación
9083,UT SOLUCIONES CC CONECTIVIDAD NACIONAL,1.53,0.00,-1.53,Saliente
1038,ENTerritorio S.A,1.48,0.04,-1.45,Perdió participación
4356,UNION TEMPORAL ALIANZA EDUCATIVA UTP - SUEJE,1.19,0.00,-1.19,Saliente
3785,CONSORCIO SAN FELIPEE,1.04,0.00,-1.04,Saliente
9110,VALOR+ S.A.S-PROVEEDOR,2.07,1.06,-1.01,Perdió participación
891,UNIVERSIDAD DE PAMPLONA,0.97,0.01,-0.96,Perdió participación
8762,SUPPLA.S.A,0.87,0.00,-0.87,Saliente


**Nota:** Los valores positivos representan proveedores que ganaron participación; los negativos indican pérdida. En el caso de los entrantes, no tenían participación en 2024 y sí aparecen en 2025, mientras que los salientes aparecen en 2024 pero no en 2025.

**Observación:** Entre 2024 y 2025 se observa una redistribución importante de la participación de mercado entre los proveedores. POr lo que, GECELCA S.A. E.S.P. aparece como el principal entrante, con un aumento de aproximadamente 23,15 puntos porcentuales, muy por encima del resto. AHora bien, también ganan participación proveedores como el Consorcio Aeropuerto del Café, la Empresa Distribuidora del Pacífico y la Agencia Nacional Inmobiliaria Virgilio Barco. En contraste, el Fondo para el Financiamiento del Sector Agropecuario, la Financiera de Desarrollo Territorial y la Empresa de Desarrollo Urbano de Medellín reducen su participación.

Con base en lo anterior, el mercado presenta cambios fuertes de un año a otro, ya que, entran nuevos proveedores con contratos de gran valor, mientras otros pierden peso o dejan de registrar adjudicaciones.

La comparación mide participación en el valor total adjudicado, no en la cantidad de procesos.

## B.2.1 Comparación homogénea de proveedores: enero–julio de 2024, 2025 y 2026

Para evitar comparar años completos con un periodo incompleto, se utiliza como fecha de corte el último día disponible de 2026. Posteriormente, se aplica el mismo día y mes a 2024 y 2025.

El análisis permite observar la evolución de la participación de los proveedores en el valor total adjudicado durante periodos equivalentes.

In [11]:
ANIOS = [2024, 2025, 2026]

# Última fecha disponible de 2026
fecha_corte_2026 = (
    df_mercado.loc[
        df_mercado["fecha_de_publicacion_del"].dt.year == 2026,
        "fecha_de_publicacion_del"
    ]
    .max()
)

mes_corte = fecha_corte_2026.month
dia_corte = fecha_corte_2026.day

print(
    f"Periodo comparado: 1 de enero al "
    f"{dia_corte:02d}/{mes_corte:02d} de cada año"
)

# Seleccionar el mismo periodo para los tres años
bases_periodo = []

for anio in ANIOS:
    fecha_inicio = pd.Timestamp(anio, 1, 1)
    fecha_fin = pd.Timestamp(anio, mes_corte, dia_corte)

    temporal = df_mercado[
        df_mercado["fecha_de_publicacion_del"].between(
            fecha_inicio,
            fecha_fin
        )
    ].copy()

    temporal["anio"] = anio
    bases_periodo.append(temporal)

base_comparable = pd.concat(
    bases_periodo,
    ignore_index=True
)

# Valor adjudicado por proveedor y año
participacion_proveedor = (
    base_comparable
    .groupby(
        ["anio", "proveedor_id"],
        as_index=False
    )
    .agg(
        proveedor=("nombre_del_proveedor", "first"),
        valor_adjudicado=(
            "valor_total_adjudicacion",
            "sum"
        ),
        cantidad_procesos=(
            "id_del_proceso",
            "nunique"
        )
    )
)

# Participación de mercado dentro de cada año
participacion_proveedor["participacion_pct"] = (
    participacion_proveedor["valor_adjudicado"]
    / participacion_proveedor.groupby("anio")[
        "valor_adjudicado"
    ].transform("sum")
    * 100
)

# Tabla con los años como columnas
evolucion_proveedores = (
    participacion_proveedor
    .pivot_table(
        index=["proveedor_id", "proveedor"],
        columns="anio",
        values="participacion_pct",
        fill_value=0
    )
    .reset_index()
)

# Garantizar que existan las tres columnas
for anio in ANIOS:
    if anio not in evolucion_proveedores.columns:
        evolucion_proveedores[anio] = 0

# Cambios en puntos porcentuales
evolucion_proveedores["cambio_2024_2025_pp"] = (
    evolucion_proveedores[2025]
    - evolucion_proveedores[2024]
)

evolucion_proveedores["cambio_2025_2026_pp"] = (
    evolucion_proveedores[2026]
    - evolucion_proveedores[2025]
)

evolucion_proveedores["cambio_2024_2026_pp"] = (
    evolucion_proveedores[2026]
    - evolucion_proveedores[2024]
)

# Clasificación entre 2025 y 2026
evolucion_proveedores["situacion_2026"] = np.select(
    [
        (evolucion_proveedores[2025] == 0)
        & (evolucion_proveedores[2026] > 0),

        (evolucion_proveedores[2025] > 0)
        & (evolucion_proveedores[2026] == 0),

        evolucion_proveedores["cambio_2025_2026_pp"] > 0
    ],
    [
        "Entrante en 2026",
        "Saliente en 2026",
        "Ganó participación"
    ],
    default="Perdió participación"
)

# Principales cambios entre 2025 y 2026
ganadores_2026 = evolucion_proveedores.nlargest(
    10,
    "cambio_2025_2026_pp"
)

perdedores_2026 = evolucion_proveedores.nsmallest(
    10,
    "cambio_2025_2026_pp"
)

print("\nPROVEEDORES QUE MÁS GANARON PARTICIPACIÓN EN 2026")
display(
    ganadores_2026[
        [
            "proveedor",
            2024,
            2025,
            2026,
            "cambio_2025_2026_pp",
            "situacion_2026"
        ]
    ]
)

print("\nPROVEEDORES QUE MÁS PERDIERON PARTICIPACIÓN EN 2026")
display(
    perdedores_2026[
        [
            "proveedor",
            2024,
            2025,
            2026,
            "cambio_2025_2026_pp",
            "situacion_2026"
        ]
    ]
)

Periodo comparado: 1 de enero al 29/07 de cada año

PROVEEDORES QUE MÁS GANARON PARTICIPACIÓN EN 2026


anio,proveedor,2024,2025,2026,cambio_2025_2026_pp,situacion_2026
4251,CONSORCIO PLANTA CURVAL,0.00,0.00,9.63,9.63,Entrante en 2026
4451,CSJT 42  HPCM.,0.00,0.00,4.18,4.18,Entrante en 2026
3431,UNIÓN TEMPORAL APOYO INTEGRAL A LA ADMINISTRAC...,0.00,0.00,3.00,3.00,Entrante en 2026
3362,DESARROLLO CONJUNTO TERRITORIAL - DETERCONSA,0.00,0.00,2.35,2.35,Entrante en 2026
3709,C. PULSO NUTRICIONAL,0.00,0.00,1.87,1.87,Entrante en 2026
4319,CONSORCIO TORRE CLINICA YOLOMBÓ 2026,0.00,0.00,1.85,1.85,Entrante en 2026
3401,CONSORCIO ALBANIA SOLAR 2026,0.00,0.00,1.77,1.77,Entrante en 2026
4310,CONSORCIO SOLUCIONES SOLARES INTEGRALES 2026,0.00,0.00,1.76,1.76,Entrante en 2026
3959,CONSORCIO ESTANQUILLO CLA,0.00,0.00,1.63,1.63,Entrante en 2026
313,CONSEJO REGIONAL INDIGENA DEL CAUCA,0.00,0.00,1.60,1.60,Entrante en 2026



PROVEEDORES QUE MÁS PERDIERON PARTICIPACIÓN EN 2026


anio,proveedor,2024,2025,2026,cambio_2025_2026_pp,situacion_2026
1184,AGENCIA NACIONAL INMOBILIARIA VIRGILIO BARCO V...,0.00,5.33,0.00,-5.33,Saliente en 2026
702,ESU,0.58,5.50,1.31,-4.19,Perdió participación
6349,VALOR+ S.A.S-PROVEEDOR,4.51,3.68,0.01,-3.67,Perdió participación
158,INSTITUCIÓN UNIVERSITARIA ITM,5.03,4.98,2.15,-2.83,Perdió participación
162,EMPRESA DE DESARROLLO URBANO DE MEDELLIN,0.06,2.43,0.13,-2.30,Perdió participación
2294,"AGENCIA DISTRITAL PARA LA EDUCACIÓN SUPERIOR, ...",0.02,2.24,0.00,-2.24,Saliente en 2026
3099,CONSORCIO ME CAROLINA,0.00,1.98,0.00,-1.98,Saliente en 2026
3945,CONSORCIO DORADA CSLE,0.00,1.67,0.00,-1.67,Saliente en 2026
677,EMPRESAS PUBLICAS DE MEDELLIN E.S.P.,2.53,1.61,0.00,-1.61,Saliente en 2026
691,UNIVERSIDAD DE ANTIOQUIA,1.02,1.51,0.26,-1.25,Perdió participación


In [12]:
# 15 proveedores con mayor participación acumulada
evolucion_proveedores["participacion_acumulada_tres_anios"] = (
    evolucion_proveedores[2024]
    + evolucion_proveedores[2025]
    + evolucion_proveedores[2026]
)

top_proveedores = (
    evolucion_proveedores
    .nlargest(
        15,
        "participacion_acumulada_tres_anios"
    )
)

datos_grafica = (
    top_proveedores[
        ["proveedor", 2024, 2025, 2026]
    ]
    .melt(
        id_vars="proveedor",
        var_name="anio",
        value_name="participacion_pct"
    )
)

fig = go.Figure()

for proveedor in datos_grafica["proveedor"].unique():
    datos_proveedor = datos_grafica[
        datos_grafica["proveedor"] == proveedor
    ]

    fig.add_trace(
        go.Scatter(
            x=datos_proveedor["anio"],
            y=datos_proveedor["participacion_pct"],
            mode="lines+markers",
            name=proveedor,
            hovertemplate=(
                "<b>%{fullData.name}</b><br>"
                "Año: %{x}<br>"
                "Participación: %{y:.2f}%"
                "<extra></extra>"
            )
        )
    )

fig.update_layout(
    title=(
        "Evolución de la participación de los principales "
        "proveedores — periodos comparables"
    ),
    xaxis_title="Año",
    yaxis_title="Participación en el valor adjudicado (%)",
    xaxis=dict(
        tickmode="array",
        tickvals=[2024, 2025, 2026]
    ),
    hovermode="x unified",
    height=700,
    template="plotly_white"
)

fig.show()

**Observación:** La comparación enero–julio muestra una alta rotación en la participación de los proveedores. Debido a que en el año 2026 aparecen nuevos participantes con aumentos importantes, especialmente Consorcio Planta Curval, que alcanza cerca del 9,63 % del valor adjudicado.

Al mismo tiempo, proveedores que tenían una participación relevante en 2024 o 2025 pierden peso o desaparecen en 2026, como la Agencia Nacional Inmobiliaria Virgilio Barco, ESU y VALOR+.

En conclusión, el mercado cambia considerablemente entre años. Ya que, no siempre lideran los mismos proveedores y una parte importante del valor adjudicado se redistribuye hacia nuevos participantes.




## B.3 Relación entre entidades y proveedores

Este análisis busca identificar las relaciones comerciales más importantes entre las entidades contratantes y sus proveedores.

Se estudian:

- El valor y número de procesos de cada relación entidad–proveedor.
- Los principales proveedores de cada entidad.
- La participación de cada proveedor dentro del valor contratado por una entidad.
- La dependencia de cada proveedor respecto a su entidad principal.
- El nivel de diversificación de los proveedores.

In [13]:
# Base para el análisis entidad–proveedor
base_relaciones = df_mercado.dropna(
    subset=["entidad", "proveedor_id"]
).copy()

# Nombre visible del proveedor
base_relaciones["proveedor_visible"] = (
    base_relaciones["nombre_del_proveedor"]
    .fillna(base_relaciones["proveedor_id"].astype("string"))
)

# Consolidar cada relación entidad–proveedor
relaciones_ep = (
    base_relaciones
    .groupby(
        ["entidad", "proveedor_id"],
        as_index=False
    )
    .agg(
        proveedor=("proveedor_visible", "first"),
        cantidad_procesos=("id_del_proceso", "nunique"),
        valor_adjudicado=("valor_total_adjudicacion", "sum")
    )
)

# Participación del proveedor dentro de cada entidad
relaciones_ep["participacion_entidad_pct"] = (
    relaciones_ep["valor_adjudicado"]
    / relaciones_ep.groupby("entidad")[
        "valor_adjudicado"
    ].transform("sum")
    * 100
)

# Dependencia del proveedor respecto a cada entidad
relaciones_ep["dependencia_proveedor_pct"] = (
    relaciones_ep["valor_adjudicado"]
    / relaciones_ep.groupby("proveedor_id")[
        "valor_adjudicado"
    ].transform("sum")
    * 100
)

# Relaciones de mayor valor
top_relaciones = (
    relaciones_ep
    .sort_values("valor_adjudicado", ascending=False)
    .head(20)
)

print("RELACIONES ENTIDAD–PROVEEDOR DE MAYOR VALOR")

display(
    top_relaciones[
        [
            "entidad",
            "proveedor",
            "cantidad_procesos",
            "valor_adjudicado",
            "participacion_entidad_pct",
            "dependencia_proveedor_pct"
        ]
    ].round(2)
)

RELACIONES ENTIDAD–PROVEEDOR DE MAYOR VALOR


,entidad,proveedor,cantidad_procesos,valor_adjudicado,participacion_entidad_pct,dependencia_proveedor_pct
18864,MINISTERIO DE MINAS Y ENERGIA,GECELCA S.A. E.S.P.,1,4205027751839,86.95,100.00
10237,DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNO...,EMPRESA DE DESARROLLO URBANO DE MEDELLIN,37,1803013119308,31.48,98.44
18549,MINISTERIO DE AGRICULTURA Y DESARROLLO RURAL,FONDO PARA EL FINANCIAMIENTO DEL SECTOR AGROPE...,4,823463605046,82.56,100.00
10236,DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNO...,INSTITUCIÓN UNIVERSITARIA ITM,149,693040277953,12.10,83.59
23588,PATRIMONIO AUTÓNOMO AEROCAFÉ,CONSORCIO AEROPUERTO DEL CAFE SK,1,634275135745,96.86,100.00
10286,DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNO...,ESU,42,577922841000,10.09,98.54
20370,MUNICIPIO DE ITAGUI,AGENCIA DE DESARROLLO LOCAL DE ITAGUI,38,465912019047,90.12,100.00
10273,DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNO...,EMPRESAS PUBLICAS DE MEDELLIN E.S.P.,14,442350629342,7.72,99.88
12000,Empresa Distrital de Desarrollo y Renovación U...,CONSORCIO PLANTA CURVAL,1,410758596204,93.27,100.00
8750,DEPARTAMENTO DE ANTIOQUIA//,RENTING DE ANTIOQUIA,21,388166326076,31.33,99.09


In [14]:
# Tres proveedores con mayor valor adjudicado por entidad
top3_por_entidad = (
    relaciones_ep
    .sort_values(
        ["entidad", "valor_adjudicado"],
        ascending=[True, False]
    )
    .groupby("entidad", group_keys=False)
    .head(3)
    .reset_index(drop=True)
)

print("TRES PRINCIPALES PROVEEDORES POR ENTIDAD")

display(
    top3_por_entidad[
        [
            "entidad",
            "proveedor",
            "cantidad_procesos",
            "valor_adjudicado",
            "participacion_entidad_pct"
        ]
    ].round(2)
)

TRES PRINCIPALES PROVEEDORES POR ENTIDAD


,entidad,proveedor,cantidad_procesos,valor_adjudicado,participacion_entidad_pct
0,(Secretaría Distrital de Integración Social),FUNDACASTILLO,4,20730556677,12.10
1,(Secretaría Distrital de Integración Social),FUNDACION PEPASO,5,15807792627,9.23
2,(Secretaría Distrital de Integración Social),MANOSUNID,4,10996800632,6.42
3,ACI Medellín,INTERNATIONAL PROJECT MANAGEMENT S.A.S.,4,75523806,61.09
4,ACI Medellín,GRM COLOMBIA S.A.S.,1,12210656,9.88
...,...,...,...,...,...
5442,sutamarchan,MILAN INGENIERÍA Y ARQUITECTURA S.A.S.,1,37223224,12.10
5443,sutamarchan,LUIS ALFREDO PÁEZ ALAYÓN,2,36402750,11.83
5444,ÁREA METROPOLITANA DE BARRANQUILLA,BUSINESS CENTER WAL SAS,2,81161367,32.18
5445,ÁREA METROPOLITANA DE BARRANQUILLA,HOSTDIME.COM.CO S.A.S,3,72500001,28.75


In [15]:
# Identificar la entidad principal de cada proveedor
entidad_principal = (
    relaciones_ep
    .sort_values(
        ["proveedor_id", "valor_adjudicado"],
        ascending=[True, False]
    )
    .drop_duplicates("proveedor_id")
    [
        [
            "proveedor_id",
            "entidad"
        ]
    ]
    .rename(
        columns={
            "entidad": "entidad_principal"
        }
    )
)

# Crear perfil de cada proveedor
perfil_proveedores = (
    relaciones_ep
    .groupby(
        ["proveedor_id", "proveedor"],
        as_index=False
    )
    .agg(
        entidades_cliente=("entidad", "nunique"),
        cantidad_procesos=("cantidad_procesos", "sum"),
        valor_total_adjudicado=("valor_adjudicado", "sum"),
        dependencia_maxima_pct=(
            "dependencia_proveedor_pct",
            "max"
        )
    )
    .merge(
        entidad_principal,
        on="proveedor_id",
        how="left"
    )
)

# Clasificación según dependencia
perfil_proveedores["perfil"] = np.select(
    [
        perfil_proveedores["dependencia_maxima_pct"] >= 80,
        perfil_proveedores["dependencia_maxima_pct"] >= 50
    ],
    [
        "Alta dependencia",
        "Dependencia moderada"
    ],
    default="Diversificado"
)

print("PERFIL DE LOS PRINCIPALES PROVEEDORES")

display(
    perfil_proveedores
    .sort_values(
        "valor_total_adjudicado",
        ascending=False
    )
    .head(20)
    .round(2)
)

PERFIL DE LOS PRINCIPALES PROVEEDORES


,proveedor_id,proveedor,entidades_cliente,cantidad_procesos,valor_total_adjudicado,dependencia_maxima_pct,entidad_principal,perfil
1706,900082143,GECELCA S.A. E.S.P.,1,1,4205027751839,100.00,MINISTERIO DE MINAS Y ENERGIA,Alta dependencia
378,800223337,EMPRESA DE DESARROLLO URBANO DE MEDELLIN,5,41,1831659864317,98.44,DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNO...,Alta dependencia
234,800096329,FINANCIERA DE DESARROLLO TERRITORIAL S.A.,13,22,1002979246316,29.33,MINISTERIO DE EDUCACION NACIONAL (MEN),Diversificado
367,800214750,INSTITUCIÓN UNIVERSITARIA ITM,4,177,829071782294,83.59,DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNO...,Alta dependencia
271,800116398,FONDO PARA EL FINANCIAMIENTO DEL SECTOR AGROPE...,1,4,823463605046,100.00,MINISTERIO DE AGRICULTURA Y DESARROLLO RURAL,Alta dependencia
4710,901508361,"AGENCIA DISTRITAL PARA LA EDUCACIÓN SUPERIOR, ...",23,79,722587713558,46.70,FONDO FINANCIERO DISTRITAL DE SALUD..,Diversificado
9083,CONSORCIO AEROPUERTO DEL CAFE SK,CONSORCIO AEROPUERTO DEL CAFE SK,1,1,634275135745,100.00,PATRIMONIO AUTÓNOMO AEROCAFÉ,Alta dependencia
1447,890984761,ESU,5,50,586464020823,98.54,DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNO...,Alta dependencia
1569,899999316,ENTerritorio S.A,7,7,502326602956,72.42,AEROCIVIL,Dependencia moderada
2688,900590434,AGENCIA DE DESARROLLO LOCAL DE ITAGUI,1,38,465912019047,100.00,MUNICIPIO DE ITAGUI,Alta dependencia


**Nota**.

La clasificación significa:

- Alta dependencia: Al menos el 80 % del valor obtenido proviene de una sola entidad.

- Dependencia moderada: Entre el 50 % y el 80 %.

- Diversificado: Ninguna entidad representa el 50 % o más.

In [16]:
resumen_perfiles = (
    perfil_proveedores["perfil"]
    .value_counts()
    .rename_axis("perfil")
    .reset_index(name="cantidad_proveedores")
)

resumen_perfiles["porcentaje"] = (
    resumen_perfiles["cantidad_proveedores"]
    / resumen_perfiles["cantidad_proveedores"].sum()
    * 100
).round(2)

display(resumen_perfiles)

,perfil,cantidad_proveedores,porcentaje
0,Alta dependencia,14629,84.82
1,Dependencia moderada,1714,9.94
2,Diversificado,904,5.24


In [17]:
# Entidades y proveedores con mayor valor
top_entidades = (
    relaciones_ep
    .groupby("entidad")["valor_adjudicado"]
    .sum()
    .nlargest(10)
    .index
)

top_proveedores = (
    relaciones_ep
    .groupby("proveedor")["valor_adjudicado"]
    .sum()
    .nlargest(12)
    .index
)

# Matriz de participación
matriz_relaciones = (
    relaciones_ep[
        relaciones_ep["entidad"].isin(top_entidades)
        & relaciones_ep["proveedor"].isin(top_proveedores)
    ]
    .pivot_table(
        index="entidad",
        columns="proveedor",
        values="participacion_entidad_pct",
        aggfunc="sum",
        fill_value=0
    )
)

fig = go.Figure(
    go.Heatmap(
        z=matriz_relaciones.values,
        x=matriz_relaciones.columns,
        y=matriz_relaciones.index,
        colorscale="Blues",
        colorbar={
            "title": "Participación (%)"
        },
        hovertemplate=(
            "<b>Entidad:</b> %{y}<br>"
            "<b>Proveedor:</b> %{x}<br>"
            "<b>Participación:</b> %{z:.2f}%"
            "<extra></extra>"
        )
    )
)

fig.update_layout(
    title=(
        "Relación entidad–proveedor por participación "
        "en el valor adjudicado"
    ),
    xaxis_title="Proveedor",
    yaxis_title="Entidad",
    height=700,
    template="plotly_white",
    margin=dict(
        l=220,
        r=50,
        t=90,
        b=230
    )
)

fig.update_xaxes(tickangle=-45)

fig.show()

# **C. Picos de enero: ¿estacionalidad real o artefacto?**

Enero 2022 (23,499) coincide con el borde del filtro de fecha (>= 2022-01-01), y
enero 2026 (30,298) duplica cualquier mes normal. Se revisa la distribucion POR DIA
dentro de esos meses: si la masa esta concentrada en el dia 1, es un artefacto de
carga/registro masivo, no comportamiento real de publicacion.

In [18]:
df_proc["fecha_de_publicacion_del"] = pd.to_datetime(
    df_proc["fecha_de_publicacion_del"],
    errors="coerce"
)

df_fechas = df_proc[df_proc["fecha_de_publicacion_del"].notna()].copy()

for anio in [2022, 2023, 2024, 2025, 2026]:
    enero = df_fechas[
        (df_fechas["fecha_de_publicacion_del"].dt.year == anio)
        & (df_fechas["fecha_de_publicacion_del"].dt.month == 1)
    ]
    if len(enero) == 0: continue
    por_dia = enero["fecha_de_publicacion_del"].dt.day.value_counts().sort_index()
    dia_max = por_dia.idxmax()
    print(f"Enero {anio}: {len(enero):>6,} procesos | dia con mas procesos: {dia_max} "
          f"({por_dia.max():,} = {por_dia.max()/len(enero):.0%} del mes)")

Enero 2022: 23,499 procesos | dia con mas procesos: 28 (2,348 = 10% del mes)
Enero 2023: 11,549 procesos | dia con mas procesos: 31 (920 = 8% del mes)
Enero 2024:  9,602 procesos | dia con mas procesos: 31 (739 = 8% del mes)
Enero 2025: 11,781 procesos | dia con mas procesos: 17 (782 = 7% del mes)
Enero 2026: 30,298 procesos | dia con mas procesos: 30 (2,541 = 8% del mes)


In [19]:
# Vista grafica de los eneros anomalos vs uno normal
fig = make_subplots(rows=1, cols=3, subplot_titles=["Enero 2022", "Enero 2024 (normal)", "Enero 2026"])
for j, anio in enumerate([2022, 2024, 2026], start=1):
    enero = df_fechas[
        (df_fechas["fecha_de_publicacion_del"].dt.year == anio)
        & (df_fechas["fecha_de_publicacion_del"].dt.month == 1)
    ]
    por_dia = enero["fecha_de_publicacion_del"].dt.day.value_counts().sort_index()
    fig.add_trace(go.Bar(x=por_dia.index, y=por_dia.values, showlegend=False), row=1, col=j)
fig.update_layout(title="Procesos por dia del mes en eneros seleccionados", template="plotly_white")
fig.show()

In [20]:
# Estacionalidad presentable: solo anios completos y estables (2023-2025)
df_estable = df_fechas[df_fechas["fecha_de_publicacion_del"].dt.year.isin([2023, 2024, 2025])]
orden_meses = ["Enero","Febrero","Marzo","Abril","Mayo","Junio",
               "Julio","Agosto","Septiembre","Octubre","Noviembre","Diciembre"]
por_mes = (
    df_estable["fecha_de_publicacion_del"].dt.month
    .map(dict(enumerate(orden_meses, start=1)))
    .value_counts().reindex(orden_meses)
)
fig = go.Figure(go.Bar(x=por_mes.index, y=por_mes.values))
fig.update_layout(title="Estacionalidad mensual 2023-2025 (anios completos y estables)",
                   template="plotly_white")
fig.show()

## C.1 Hallazgos — estacionalidad

- **Los picos de enero 2022 y 2026 NO son artefactos de carga masiva**: si fueran una
  recarga/duplicacion concentrada en un solo dia, se veria un dia acumulando una fraccion
  enorme del mes. No es el caso — el dia con mas procesos nunca supera el 10% del mes en
  ningun enero (2022: dia 28 con 10%; 2023: dia 31 con 8%; 2024: dia 31 con 8%; 2025: dia
  17 con 7%; 2026: dia 30 con 8%), practicamente el mismo nivel de concentracion diaria
  que los eneros "normales" 2023-2025. **Esto revierte la sospecha planteada en el EDA
  anterior**: el volumen extra de enero 2022 y 2026 esta distribuido a lo largo del mes,
  consistente con actividad real de publicacion, no con un evento de carga puntual.
- Dicho esto, la magnitud si sigue siendo atipica y queda sin explicar del todo: enero
  2026 (30,298) es ~2.6x el promedio de enero 2023-2025 (~11,270) y enero 2022 (23,499)
  es ~2.1x ese mismo promedio. Como no es un artefacto de un solo dia, hay dos hipotesis
  reales que valdria la pena documentar como limitacion (no se puede resolver solo con
  este dataset): (a) 2022 es el primer anio del dataset y puede incluir una acumulacion
  de procesos "atrasados" que se publicaron todos al arrancar el tracking, y (b) 2026 es
  el anio mas reciente y podria reflejar un cambio real de politica/ciclo presupuestal o
  mayor digitalizacion de entidades — no hay forma de distinguir ambas con los datos
  disponibles.
- **La estacionalidad de referencia queda bien establecida usando solo 2023-2025**: son
  3 anios completos y con volumen mensual estable (sin el sesgo de arranque de 2022 ni el
  corte incompleto de 2026), la base correcta para cualquier afirmacion de "estacionalidad
  tipica" en el reporte.

# **D. Cierre de preparación para la Capacidad 3 (sin entrenar modelos)**

## D.1 Dos universos de modelado, no uno

| Tarea del reto | Universo | Justificación |
|---|---|---|
| Probabilidad de adjudicación | Solo modalidades **competitivas** | En régimen especial y contratación directa, el campo `adjudicado_proceso` no representa una competencia comparable |
| Tipo de contratación futura, rangos de presupuesto y sectores con mayor inversión | **Todos** los procesos válidos | Estas predicciones no dependen del campo de adjudicación |


In [21]:
MODALIDADES_COMPETITIVAS = [
    "Licitación pública", "Licitación pública Obra Publica",
    "Licitación Pública Acuerdo Marco de Precios",
    "Concurso de méritos abierto", "Concurso de méritos con precalificación",
    "Selección Abreviada de Menor Cuantía",
    "Seleccion Abreviada Menor Cuantia Sin Manifestacion Interes",
    "Selección abreviada subasta inversa", "Mínima cuantía",
    "Contratación Directa (con ofertas)", "Contratación régimen especial (con ofertas)",
]
df_competitivo = df_proc[df_proc["modalidad_de_contratacion"].isin(MODALIDADES_COMPETITIVAS)].copy()
print(f"Universo competitivo: {len(df_competitivo):,} procesos "
      f"({len(df_competitivo)/len(df_proc):.1%} del total)")
print(f"Tasa de adjudicacion en el universo competitivo: "
      f"{df_competitivo['adjudicado_proceso'].mean():.1%}")
print()
print(df_competitivo.groupby("modalidad_de_contratacion")["adjudicado_proceso"]
      .agg(tasa="mean", n="size").assign(tasa=lambda d: (d["tasa"]*100).round(1))
      .sort_values("n", ascending=False))

Universo competitivo: 72,236 procesos (14.7% del total)
Tasa de adjudicacion en el universo competitivo: 53.8%

                                                    tasa      n
modalidad_de_contratacion                                      
Mínima cuantía                                     77.70  22685
Concurso de méritos abierto                        39.90  16755
Selección Abreviada de Menor Cuantía                9.80  10464
Contratación Directa (con ofertas)                 78.40   8210
Contratación régimen especial (con ofertas)        66.40   5686
Selección abreviada subasta inversa                40.10   4909
Licitación pública                                 39.60   2134
Licitación pública Obra Publica                    39.50   1191
Seleccion Abreviada Menor Cuantia Sin Manifesta... 37.20    156
Concurso de méritos con precalificación            20.50     39
Licitación Pública Acuerdo Marco de Precios        42.90      7


## D.2 Clasificación de variables por momento de disponibilidad

Momento de predicción definido: **al publicarse el proceso**.

| Variable | ¿Disponible al publicar? | Decisión |
|---|---|---|
| Segmento/familia UNSPSC, modalidad, tipo de contrato, entidad, territorio, precio base, duración, número de lotes, mes/año de publicación y texto del procedimiento | Sí | Variable candidata |
| Historial agregado de la entidad o proveedor calculado únicamente con datos anteriores al proceso | Sí | Variable candidata con ventana temporal |
| `respuestas_al_procedimiento`, `conteo_de_respuestas_a_ofertas` | No, se conocen al cierre | Excluir por fuga de información |
| `proveedores_con_invitacion`, `proveedores_unicos_con` | Disponibilidad y significado variables según la modalidad | Excluir del modelo global; conservar para análisis descriptivo y evaluar solo en modelos específicos por modalidad |
| `valor_total_adjudicacion`, `valor_adjudicado_total`, fecha de adjudicación y proveedor adjudicado | No, corresponden al resultado | Excluir como predictores contemporáneos |

La decisión definitiva sobre las variables de invitación se valida en la sección D.4.


In [22]:
# Verificacion rapida del split temporal sobre el universo competitivo
df_comp_fecha = df_competitivo[df_competitivo["fecha_de_publicacion_del"].notna()].copy()
FECHA_CORTE_SPLIT = "2025-07-01"
train = df_comp_fecha[df_comp_fecha["fecha_de_publicacion_del"] < FECHA_CORTE_SPLIT]
test = df_comp_fecha[df_comp_fecha["fecha_de_publicacion_del"] >= FECHA_CORTE_SPLIT]
print(f"Universo competitivo — Train: {len(train):,} | Test: {len(test):,}")
print(f"Tasa adjudicacion train: {train['adjudicado_proceso'].mean():.1%} | "
      f"test: {test['adjudicado_proceso'].mean():.1%}")

Universo competitivo — Train: 52,043 | Test: 17,641
Tasa adjudicacion train: 55.4% | test: 55.1%


In [23]:
"""
Deflactación de montos COP a pesos constantes
==============================================

Regla metodológica:
1. Identificar primero los procesos afectados por valores monetarios implausibles.
2. Excluir esos procesos del universo monetario.
3. Aplicar el factor IPC únicamente sobre los registros plausibles.

La base completa de procesos se conserva para análisis no monetarios. Para tareas que
usan presupuesto o valor adjudicado se emplea `df_proc_deflactado`.
"""

RUTA_IPC = Path("ipc_dane_mensual_interpolado_TOTAL.csv")
BASE = "2026-06"


def cargar_ipc(path: Path = RUTA_IPC) -> pd.DataFrame:
    ipc = pd.read_csv(path)
    ipc["anio_mes"] = pd.to_datetime(
        ipc["anio"].astype(str)
        + "-"
        + ipc["mes"].astype(str).str.zfill(2)
        + "-01",
        errors="coerce"
    )

    ipc["indice"] = pd.to_numeric(ipc["indice"], errors="coerce")

    ipc = (
        ipc[["anio_mes", "indice"]]
        .dropna(subset=["anio_mes", "indice"])
        .drop_duplicates("anio_mes")
        .sort_values("anio_mes")
        .reset_index(drop=True)
    )

    if ipc.empty:
        raise ValueError("La tabla de IPC quedó vacía después de la validación.")

    return ipc


def construir_deflactor(ipc: pd.DataFrame, base: str = BASE) -> pd.DataFrame:
    periodo_base = pd.Timestamp(base + "-01")
    indice_base = ipc.loc[ipc["anio_mes"].eq(periodo_base), "indice"]

    if indice_base.empty:
        raise ValueError(
            f"No hay un índice IPC para el periodo base {base}. "
            "Verifica la cobertura del archivo."
        )

    out = ipc.copy()
    out["factor_deflactor"] = indice_base.iloc[0] / out["indice"]
    return out


def deflactar(
    df: pd.DataFrame,
    col_fecha: str,
    columnas_monetarias: list[str],
    deflactor: pd.DataFrame
) -> pd.DataFrame:
    """Agrega columnas `<variable>_real` expresadas en pesos del periodo BASE."""

    columnas_faltantes = [
        col for col in [col_fecha, *columnas_monetarias]
        if col not in df.columns
    ]
    if columnas_faltantes:
        raise KeyError(
            f"Faltan columnas necesarias para deflactar: {columnas_faltantes}"
        )

    out = df.copy()
    out[col_fecha] = pd.to_datetime(out[col_fecha], errors="coerce")
    out["_anio_mes"] = out[col_fecha].dt.to_period("M").dt.to_timestamp()

    out = out.merge(
        deflactor[["anio_mes", "factor_deflactor"]],
        left_on="_anio_mes",
        right_on="anio_mes",
        how="left",
        validate="many_to_one"
    )

    for col in columnas_monetarias:
        out[col] = pd.to_numeric(out[col], errors="coerce")
        out[f"{col}_real"] = out[col] * out["factor_deflactor"]

    return out.drop(columns=["_anio_mes", "anio_mes"])



# 1. Llevar la marca de implausibilidad desde líneas hacia procesos


if "flag_valor_implausible" not in df_lineas.columns:
    raise RuntimeError(
        "Primero ejecuta la sección A, donde se crea `flag_valor_implausible`."
    )

ids_implausibles_lineas = set(
    df_lineas.loc[
        df_lineas["flag_valor_implausible"],
        "id_del_proceso"
    ].dropna()
)

df_proc_preparado = df_proc.copy()

flag_por_lineas = df_proc_preparado["id_del_proceso"].isin(
    ids_implausibles_lineas
)

# Validación adicional directamente en el agregado por proceso

valor_proc = pd.to_numeric(
    df_proc_preparado["valor_adjudicado_total"],
    errors="coerce"
)
precio_proc = pd.to_numeric(
    df_proc_preparado["precio_base"],
    errors="coerce"
)

flag_abs_proc = valor_proc.gt(UMBRAL_ABSOLUTO)
flag_rel_proc = (
    precio_proc.gt(1e6)
    & valor_proc.gt(RATIO_MAX * precio_proc)
)

df_proc_preparado["flag_valor_implausible"] = (
    flag_por_lineas
    | flag_abs_proc
    | flag_rel_proc
)

n_total = len(df_proc_preparado)
n_implausibles = int(
    df_proc_preparado["flag_valor_implausible"].sum()
)

print(f"Procesos totales: {n_total:,}")
print(
    "Procesos excluidos del universo monetario: "
    f"{n_implausibles:,} ({n_implausibles / n_total:.4%})"
)

# La base completa queda disponible para análisis no monetarios.
# Para presupuesto y valor adjudicado se usa solo el subconjunto plausible.
df_proc_monetario = df_proc_preparado.loc[
    ~df_proc_preparado["flag_valor_implausible"]
].copy()

assert not df_proc_monetario["flag_valor_implausible"].any()



# 2. Deflactar únicamente después del filtro


ipc = cargar_ipc()
deflactor = construir_deflactor(ipc)

columnas_monetarias = [
    col
    for col in ["precio_base", "valor_adjudicado_total"]
    if col in df_proc_monetario.columns
]

df_proc_deflactado = deflactar(
    df=df_proc_monetario,
    col_fecha="fecha_de_publicacion_del",
    columnas_monetarias=columnas_monetarias,
    deflactor=deflactor
)

print()
print(f"Procesos plausibles deflactados: {len(df_proc_deflactado):,}")
print(
    "Procesos con fecha fuera de la cobertura del IPC: "
    f"{df_proc_deflactado['factor_deflactor'].isna().sum():,}"
)

columnas_revision = [
    col for col in [
        "id_del_proceso",
        "fecha_de_publicacion_del",
        "precio_base",
        "precio_base_real",
        "valor_adjudicado_total",
        "valor_adjudicado_total_real",
        "factor_deflactor"
    ]
    if col in df_proc_deflactado.columns
]

display(df_proc_deflactado[columnas_revision].head())

df_proc_deflactado.to_csv(
    "secop_ctei_procesos_deflactado_sin_implausibles.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Archivo guardado: "
    "secop_ctei_procesos_deflactado_sin_implausibles.csv"
)


Procesos totales: 492,797
Procesos excluidos del universo monetario: 13 (0.0026%)

Procesos plausibles deflactados: 492,784
Procesos con fecha fuera de la cobertura del IPC: 10,632


,id_del_proceso,fecha_de_publicacion_del,precio_base,precio_base_real,valor_adjudicado_total,valor_adjudicado_total_real,factor_deflactor
0,CO1.REQ.10000009,2026-01-28,12400000,"12,888,776.03",0,0.00,1.04
1,CO1.REQ.10000017,2026-01-30,47000000,"48,852,618.84",0,0.00,1.04
2,CO1.REQ.10000046,2026-01-28,46585387,"48,421,662.86",0,0.00,1.04
3,CO1.REQ.10000065,2026-01-28,21600000,"22,451,416.32",0,0.00,1.04
4,CO1.REQ.10000067,2026-01-28,15756300,"16,377,372.73",0,0.00,1.04


Archivo guardado: secop_ctei_procesos_deflactado_sin_implausibles.csv


## D.3 Validación del filtro previo a la deflactación

La preparación monetaria queda separada en dos bases:

- `df_proc_preparado`: conserva todos los procesos e incorpora la bandera de plausibilidad.
- `df_proc_deflactado`: contiene únicamente procesos con valores plausibles y montos expresados en pesos constantes de junio de 2026.

Esta separación evita eliminar procesos útiles para análisis no monetarios y, al mismo tiempo, impide que los valores anómalos afecten modelos de presupuesto, series económicas o comparaciones intertemporales.


In [24]:
# Diagnóstico de disponibilidad de las variables de invitación por modalidad

COLUMNAS_INVITACION_CANDIDATAS = [
    "proveedores_con_invitacion",
    "proveedores_unicos_con"
]

columnas_invitacion = [
    col
    for col in COLUMNAS_INVITACION_CANDIDATAS
    if col in df_proc_preparado.columns
]

if not columnas_invitacion:
    print(
        "No se encontraron columnas de invitación con esos nombres "
        "en df_proc_preparado."
    )
    perfil_invitacion = pd.DataFrame()
else:
    perfiles = []

    for col in columnas_invitacion:
        temporal = df_proc_preparado[
            ["modalidad_de_contratacion", "id_del_proceso", col]
        ].copy()

        temporal[col] = pd.to_numeric(
            temporal[col],
            errors="coerce"
        )

        resumen = (
            temporal
            .groupby(
                "modalidad_de_contratacion",
                dropna=False
            )
            .agg(
                procesos=("id_del_proceso", "size"),
                registros_informados=(
                    col,
                    lambda s: int(s.notna().sum())
                ),
                porcentaje_informado=(
                    col,
                    lambda s: s.notna().mean() * 100
                ),
                porcentaje_mayor_cero=(
                    col,
                    lambda s: s.fillna(0).gt(0).mean() * 100
                ),
                mediana=(
                    col,
                    "median"
                )
            )
            .reset_index()
        )

        resumen["variable"] = col
        perfiles.append(resumen)

    perfil_invitacion = pd.concat(
        perfiles,
        ignore_index=True
    )

    perfil_invitacion = perfil_invitacion[
        [
            "variable",
            "modalidad_de_contratacion",
            "procesos",
            "registros_informados",
            "porcentaje_informado",
            "porcentaje_mayor_cero",
            "mediana"
        ]
    ].sort_values(
        ["variable", "procesos"],
        ascending=[True, False]
    )

    display(
        perfil_invitacion.round(2)
    )


,variable,modalidad_de_contratacion,procesos,registros_informados,porcentaje_informado,porcentaje_mayor_cero,mediana
4,proveedores_con_invitacion,Contratación régimen especial,219248,219248,100.00,0.00,0.00
3,proveedores_con_invitacion,Contratación directa,185886,185886,100.00,0.00,0.00
10,proveedores_con_invitacion,Mínima cuantía,22685,22685,100.00,0.00,0.00
0,proveedores_con_invitacion,Concurso de méritos abierto,16755,16755,100.00,0.00,0.00
15,proveedores_con_invitacion,Solicitud de información a los Proveedores,15225,15225,100.00,10.36,0.00
13,proveedores_con_invitacion,Selección Abreviada de Menor Cuantía,10464,10464,100.00,12.47,0.00
2,proveedores_con_invitacion,Contratación Directa (con ofertas),8210,8210,100.00,92.47,1.00
5,proveedores_con_invitacion,Contratación régimen especial (con ofertas),5686,5686,100.00,0.00,0.00
14,proveedores_con_invitacion,Selección abreviada subasta inversa,4909,4909,100.00,0.00,0.00
8,proveedores_con_invitacion,Licitación pública,2134,2134,100.00,0.00,0.00


## D.4 Decisión sobre `proveedores_con_invitacion`

### Decisión final

Las variables `proveedores_con_invitacion` y `proveedores_unicos_con` se **excluyen del modelo global de probabilidad de adjudicación**.

La decisión se adopta porque:

1. Su disponibilidad y significado no son homogéneos entre modalidades.
2. Un valor nulo o igual a cero puede significar ausencia real de invitados, dato no aplicable o dato no reportado.
3. Imputarlas globalmente con cero mezclaría situaciones distintas.
4. No está garantizado que el dato tenga el mismo momento de disponibilidad en todos los procesos, lo que introduce riesgo de fuga de información.

Estas variables se conservarán para análisis descriptivos. Solo podrían incorporarse posteriormente en modelos separados por modalidad cuando se verifique que:

- La modalidad utiliza formalmente invitaciones.
- El dato está disponible al momento de publicación.
- La cobertura dentro de esa modalidad es suficiente y estable.

Por tanto, **no se imputarán con cero ni se usarán como predictoras en el modelo global**.


In [25]:
# Listas documentadas para el futuro pipeline de modelado.

VARIABLES_POST_CIERRE = [
    "respuestas_al_procedimiento",
    "conteo_de_respuestas_a_ofertas",
    "valor_total_adjudicacion",
    "valor_adjudicado_total",
    "fecha_adjudicacion",
    "nombre_del_proveedor",
    "nit_del_proveedor_adjudicado"
]

VARIABLES_INVITACION_EXCLUIDAS_GLOBAL = [
    col
    for col in [
        "proveedores_con_invitacion",
        "proveedores_unicos_con"
    ]
    if col in df_proc_preparado.columns
]

VARIABLES_EXCLUIDAS_MODELO_GLOBAL = sorted(
    set(
        [
            col
            for col in VARIABLES_POST_CIERRE
            if col in df_proc_preparado.columns
        ]
        + VARIABLES_INVITACION_EXCLUIDAS_GLOBAL
    )
)

print("Variables excluidas del futuro modelo global:")
for variable in VARIABLES_EXCLUIDAS_MODELO_GLOBAL:
    print(f" - {variable}")

print()
print(
    "Preparación de datos cerrada. "
    "Aún no se ha iniciado el entrenamiento predictivo."
)


Variables excluidas del futuro modelo global:
 - fecha_adjudicacion
 - proveedores_con_invitacion
 - proveedores_unicos_con
 - respuestas_al_procedimiento
 - valor_adjudicado_total

Preparación de datos cerrada. Aún no se ha iniciado el entrenamiento predictivo.


## D.5 Hallazgos — cierre de preparación

- El universo competitivo queda definido para la futura predicción de adjudicación y se mantiene separado del universo completo requerido por las demás tareas del reto.
- Las variables conocidas después del cierre quedan identificadas y excluidas como predictoras contemporáneas.
- La deflactación se aplica únicamente después de excluir los procesos afectados por valores monetarios implausibles.
- La base completa se conserva para análisis no monetarios, mientras que `df_proc_deflactado` se reserva para tareas que utilizan montos.
- `proveedores_con_invitacion` y `proveedores_unicos_con` quedan excluidas del modelo global; no se imputan con cero y solo podrán evaluarse en modelos específicos por modalidad después de verificar disponibilidad temporal y cobertura.

# **E. Conclusiones para el reporte**

1. El dataset es confiable en estructura, pero cualquier análisis monetario debe aplicar primero la regla de plausibilidad.
2. El mercado CTeI es competitivo a nivel agregado, aunque existen nichos legítimos de concentración por entidad y sector.
3. La estacionalidad de enero es consistente con el ciclo presupuestal; los picos extremos deben documentarse como limitación contextual.
4. La evolución temporal y las relaciones entidad–proveedor complementan el HHI y permiten estudiar cambios de participación, dependencia y diversificación.
5. La preparación previa al modelado queda cerrada, es decir, los valores implausibles se excluyen antes de deflactar y las variables de invitación se descartan del modelo global.

